1. construct the DCOPF latex tutorial
2. check the PU in the formulation, since the 30bus is infeasible

In [7]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import json

In [22]:
file="C:\\Users\\firda\\belajar\\ISE711\\old\\DCOPF-main\\DCOPF-main\\excel_outputs\\pglib_opf_case3_lmbd.xlsx"

In [23]:
mpc_data = pd.read_excel(file, sheet_name=['baseMVA', 'bus', 'gen', 'gencost', 'branch'])

In [38]:
c1 = dict(zip(mpc_data['gencost']["gen_ID"], [mpc_data['gencost']['c2'],mpc_data['gencost']['c1'],mpc_data['gencost']['c0']]))

In [ ]:
import numpy as np
# Data preparation
# Example: load sets from mpc_data
buses = mpc_data['bus'].index.tolist()                    # list of bus IDs
gens = mpc_data['gen']['gen_ID'].tolist()                   # list of generator IDs
branches = mpc_data['branch'].index.tolist()              # assume each element is a tuple: (from_bus, to_bus)
# Parameters
# Generator cost coefficients (assume c2==0 and c0==0 for all)
# Bus power demand (MW)
Pd = mpc_data['bus']['Pd'].to_dict()
# Generator capacity limits (MW)
Pmax = dict(zip(mpc_data['gen']["gen_ID"], mpc_data['gen']['Pmax']))
Pmin = dict(zip(mpc_data['gen']["gen_ID"], mpc_data['gen']['Pmin']))
# Transmission line limits (MW)
Pmax_line = mpc_data['branch']['rateA'].to_dict()
# Line susceptance (1/X), assuming per unit values
B_param = (mpc_data['branch']['x'] /
           (mpc_data['branch']['x']**2 + mpc_data['branch']['r']**2)).to_dict()
# Generator bus assignment: map each generator to its bus.
# (Here we assume that the generator DataFrame index (or another column) gives the bus)
gen_bus = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen'].index))
# -------------------------
# Create Gurobi Model
# -------------------------
# model = gp.Model("DCOPF")

# # -------------------------
# # Decision Variables
# # -------------------------
# # Generator outputs: Pg[g] between Pmin and Pmax for each generator g
# Pg = model.addVars(gens, lb=0, name="Pg")
# for g in gens:
#     Pg[g].lb = Pmin[g]
#     Pg[g].ub = Pmax[g]

# # Voltage angles (in radians) for each bus; here we bound them between -100 and 100.
# theta = model.addVars(buses, lb=-100, ub=100, name="theta")

# # Line flows for each branch; bounds from -Pmax_line to +Pmax_line.
# P_flow = model.addVars(branches, name="P_flow")
# for l in branches:
#     P_flow[l].lb = -Pmax_line[l]
#     P_flow[l].ub = Pmax_line[l]

# # -------------------------
# # Objective Function
# # -------------------------
# # Minimize total generation cost: sum(c1[g] * Pg[g]) for all generators g.
# model.setObjective(gp.quicksum(c1[g] * Pg[g] for g in gens), GRB.MINIMIZE)

# # -------------------------
# # Constraints
# # -------------------------
# # Power Balance at each bus b:
# #   (Sum of generator outputs at bus b) +
# #   (Sum of incoming flows into b) - (Sum of outgoing flows from b)
# #   must equal the demand Pd[b].
# for b in buses:
#     expr = (gp.quicksum(Pg[g] for g in gens if gen_bus[g] == b) +
#             gp.quicksum(P_flow[l] for l in branches if l[1] == b) -
#             gp.quicksum(P_flow[l] for l in branches if l[0] == b))
#     model.addConstr(expr == Pd[b], name=f"power_balance_{b}")

# # Line Flow (DC Power Flow) constraints:
# #   For each branch l = (from_bus, to_bus),
# #   P_flow[l] == B_param[l] * (theta[from_bus] - theta[to_bus])
# for l in branches:
#     fbus, tbus = l  # unpack the tuple: from_bus and to_bus
#     model.addConstr(P_flow[l] == B_param[l] * (theta[fbus] - theta[tbus]),
#                         name=f"line_flow_{l}")

# # Reference Bus Constraint:
# #   Set the voltage angle of the reference bus to 0. (Assume bus '1' is the reference.)
# #   (Adjust if your reference bus is different.)
# model.addConstr(theta[1] == 0, name="ref_bus")

# # -------------------------
# # Optimize the Model
# # -------------------------
# # Set a tighter optimality tolerance if needed.
# model.Params.OptimalityTol = 1e-8
# model.optimize()

# # -------------------------
# # Display Results
# # -------------------------
# print("\nOptimal Generator Outputs (MW):")
# for g in gens:
#     print(f"Generator {g}: {Pg[g].X} MW")

# print("\nOptimal Line Flows (MW):")
# for l in branches:
#     print(f"Line {l}: {P_flow[l].X} MW")

# print("\nVoltage Angles (Radians):")
# for b in buses:
#     print(f"Bus {b}: {theta[b].X} rad")
